# 10.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/03-streamlit/solutions.ipynb)

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


In [2]:
scaler, logreg = bundle["pipeline"][0], bundle["pipeline"][-1]

def contributions(values):
    """Each feature's push on the log-odds: weight × standardised value."""
    z = (np.asarray(values, dtype=float) - scaler.mean_) / scaler.scale_
    return pd.Series(logreg.coef_[0] * z, index=bundle["features"])

def probability(values):
    return 1 / (1 + np.exp(-(logreg.intercept_[0] + contributions(values).sum())))

## Exercise 1

In [3]:
risky = [1, 80, 3, 273, 273]
c = contributions(risky)
print(c.round(3))
print(f"log-odds {logreg.intercept_[0] + c.sum():.3f}  →  p = {probability(risky):.3f}")

orders             0.453
spend             -0.016
products           0.332
tenure_days       -0.214
days_since_last    1.020
dtype: float64
log-odds 1.099  →  p = 0.750


Recency dominates: 273 days since the last order is more than two standard deviations above the average, and recency has the largest positive weight. Having only one order adds a little more risk. Tenure pulls slightly the other way (an older account), which is why the total is high but not extreme.

## Exercise 2

A minimal version of the new tab, to add after the existing code is moved into `with tab1:`:

```python
tab1, tab2 = st.tabs(["One customer", "Upload a CSV"])
with tab2:
    upload = st.file_uploader("Customers CSV", type="csv")
    if upload is not None:
        df = pd.read_csv(upload)
        missing = [f for f in bundle["features"] if f not in df.columns]
        if missing:
            st.error(f"Missing columns: {missing}")
        else:
            p = bundle["pipeline"].predict_proba(df[bundle["features"]])[:, 1]
            df["churn_probability"], df["voucher"] = p.round(3), p >= threshold
            st.dataframe(df)
            st.download_button("Download scores", df.to_csv(index=False), "scores.csv")
```

Test it with the CSV made in 10.1 (`models/customers_to_score.csv`). Note that rows with missing values make `predict_proba` fail: validate them first, as `score_customers.py` does.

## Exercise 3

In [4]:
from sklearn.model_selection import train_test_split
cust = cm.build_customer_table(cm.load_sales())
_, X_test, _, y_test = train_test_split(cust[cm.FEATURES], cust["churned"], test_size=0.25,
                                        random_state=42, stratify=cust["churned"])
p = bundle["pipeline"].predict_proba(X_test)[:, 1]
between = (p >= bundle["threshold"]) & (p < 5 / 60)
print(f"{between.sum()} of {len(p)} test customers fall between 0.07 and 0.083")
print(X_test[between].median().round(0))
print(f"actual churn rate among them: {y_test[between].mean():.0%}")

8 of 830 test customers fall between 0.07 and 0.083
orders                7.0
spend              4449.0
products            144.0
tenure_days         258.0
days_since_last      22.0
dtype: float64
actual churn rate among them: 12%


Only 8 customers, all very loyal (a median of 7 orders and 22 days since the last one). One of them churned. That is 12%, above the 8.3% break-even, so for this group the bundle's lower threshold happened to be right; but one customer is far too little evidence to choose a rule on. With so few customers in the gap the two rules cost almost the same. The choice matters more when vouchers are expensive.